In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Load Dataset
df = pd.read_csv("../Resources/Data/Week-2-GA-Dataset.csv")
df.head()

,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,35900,37.0,Male,Other,Strategy,23.929404,NaN,Hard,3,124,99,18,Medium
1,27085,25.0,Male,NaN,Action,22.755168,1.0,Easy,14,84,84,12,Medium
2,39595,24.0,Female,Europe,Simulation,19.505292,0.0,Hard,3,172,9,18,Medium
3,37440,26.0,Female,Europe,RPG,11.009645,NaN,NaN,3,83,36,43,Low
4,22882,17.0,Female,USA,RPG,0.581039,1.0,Medium,5,163,9,24,Medium


In [2]:
# Question 1: Which of the following columns have object datatype?
# Options: Age, Gender, Location, GameGenre, PlayTimeHours

object_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Object dtype columns:", object_columns)

Object dtype columns: ['Gender', 'Location', 'GameGenre', 'GameDifficulty', 'EngagementLevel']


In [3]:
# Question 2: In this dataset, how many "Males" from "Europe" have made "InGamePurchases"?

q2_count = ((df["Gender"] == "Male") & (df["Location"] == "Europe") & (df["InGamePurchases"] == 1)).sum()
print("Answer:", q2_count)

Answer: 299


In [4]:
# Question 3: In your dataset, how many players under the Age 18 have strictly greater than 10 PlayTimeHours?

q3_count = ((df["Age"] < 18) & (df["PlayTimeHours"] > 10)).sum()
print("Answer:", q3_count)

Answer: 453


In [5]:
# Question 4: Create X and y, then find total null values in the whole dataset.
# y = EngagementLevel, X = all remaining columns

X = df.drop(columns=["EngagementLevel"])
y = df["EngagementLevel"]

total_null_values = df.isna().sum().sum()
print("Total null values:", total_null_values)

Total null values: 3337


In [6]:
# Question 5: Split with test_size=0.2 and random_state=42.
# Which category has the least value counts in y_train?

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("y_train value counts:")
print(y_train.value_counts())
print("Least frequent category:", y_train.value_counts().idxmin())

y_train value counts:
EngagementLevel
Medium    3983
Low       2021
High      1996
Name: count, dtype: int64
Least frequent category: High


In [ ]:
# Question 6: Impute missing/Unknown values using only training-set statistics.
# Rules:
# 1) Age Unknown -> mean(Age)
# 2) Location Unknown -> "Other"
# 3) GameDifficulty Unknown -> most frequent value
# 4) InGamePurchases Unknown -> 0
# Also handle NaN values using training-derived statistics.
# Find sum of imputed Age column in test set (2 decimals).

X_train_imp = X_train.copy()
X_test_imp = X_test.copy()


age_train_valid = pd.to_numeric(
    X_train_imp["Age"].replace("Unknown", np.nan), errors="coerce"
).dropna()
age_mean = age_train_valid.mean()

gd_train_valid = X_train_imp.loc[
    X_train_imp["GameDifficulty"].notna()
    & (X_train_imp["GameDifficulty"] != "Unknown"),
    "GameDifficulty",
]
gd_mode = gd_train_valid.mode().iloc[0]


for data in [X_train_imp, X_test_imp]:
    data["Age"] = pd.to_numeric(data["Age"].replace("Unknown", np.nan), errors="coerce")
    data["Age"] = data["Age"].fillna(age_mean)

    data["Location"] = data["Location"].replace("Unknown", "Other")
    data["GameDifficulty"] = data["GameDifficulty"].replace("Unknown", gd_mode)
    data["InGamePurchases"] = data["InGamePurchases"].replace("Unknown", 0)


for col in X_train_imp.columns:
    train_col = X_train_imp[col]

    if pd.api.types.is_numeric_dtype(train_col):
        fill_value = pd.to_numeric(train_col, errors="coerce").dropna().mean()
        X_train_imp[col] = pd.to_numeric(X_train_imp[col], errors="coerce").fillna(
            fill_value
        )
        X_test_imp[col] = pd.to_numeric(X_test_imp[col], errors="coerce").fillna(
            fill_value
        )
    else:
        valid = train_col[(train_col.notna()) & (train_col != "Unknown")]
        fill_value = valid.mode().iloc[0]
        X_train_imp[col] = (
            X_train_imp[col].replace("Unknown", np.nan).fillna(fill_value)
        )
        X_test_imp[col] = X_test_imp[col].replace("Unknown", np.nan).fillna(fill_value)

print("Missing values in X_train after imputation:", X_train_imp.isna().sum().sum())
print("Missing values in X_test after imputation:", X_test_imp.isna().sum().sum())
print("Sum of imputed Age in X_test:", round(X_test_imp["Age"].sum(), 2))

Missing values in X_train after imputation: 0
Missing values in X_test after imputation: 0
Sum of imputed Age in X_test: 63585.24


In [8]:
# Question 7: Preprocess train/test features.
# Steps:
# 1) Drop PlayerID
# 2) Ordinal encode GameDifficulty: Easy=0, Medium=1, Hard=2
# 3) One-hot encode Gender, Location, GameGenre with drop_first=True
# 4) Standard scale all transformed features
# Find sum of all values in first 5 rows of transformed test feature matrix (2 decimals).

X_train_final = X_train_imp.drop(columns=["PlayerID"])
X_test_final = X_test_imp.drop(columns=["PlayerID"])

ordinal_features = ["GameDifficulty"]
nominal_features = ["Gender", "Location", "GameGenre"]
numeric_features = [
    c for c in X_train_final.columns if c not in ordinal_features + nominal_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("ord", OrdinalEncoder(categories=[["Easy", "Medium", "Hard"]]), ordinal_features),
        (
            "nom",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            nominal_features,
        ),
        ("num", "passthrough", numeric_features),
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train_final)
X_test_transformed = preprocessor.transform(X_test_final)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_transformed)
X_test_scaled = scaler.transform(X_test_transformed)

q7_sum = round(X_test_scaled[:5].sum(), 2)
print("Answer:", q7_sum)

Answer: -7.21
